# Safety Solver/Judge Case Study Extraction

This notebook extracts a few safety examples for manual inspection without displaying harmful prompt/response text by default.

Targets:

- **Mistral 14B**: prompts/items it can judge successfully in HarmMetricEval but does not solve in the safety solver matrix.
- **Qwen3.5 2B**: prompts/items it solves in the safety solver matrix but does not judge successfully in HarmMetricEval.

Important convention used here:

- In `safety_all_attacks_response_matrix.csv`, `1.0` means **attack failed / model stayed safe** and `0.0` means **attack succeeded / model did not stay safe**.
- In `harmmetric_eval_response_matrix.csv`, higher `overall_effectiveness_score` means the judge scored the prompt response as more effective according to the HarmMetricEval rubric. This is continuous; `0.5` is not automatically random guessing.

In [ ]:
from pathlib import Path
import ast
import json

import numpy as np
import pandas as pd

OUT_DIR = Path("IRT/paper_artifacts/safety/case_studies")
OUT_DIR.mkdir(parents=True, exist_ok=True)

SOLVER_MATRIX_PATH = Path("benchmarks/safety/final_solver/response_matrices/safety_all_attacks_response_matrix.csv")
SOLVER_ITEM_META_PATH = Path("benchmarks/safety/final_solver/response_matrices/safety_all_attacks_item_metadata.csv")
JUDGE_MATRIX_PATH = Path("benchmarks/HarmMetric_Eval/response_matrices/harmmetric_eval_response_matrix.csv")
JUDGE_ITEM_META_PATH = Path("benchmarks/HarmMetric_Eval/response_matrices/harmmetric_eval_item_metadata.csv")
PROMPT_LOOKUP_PATH = Path("K-Factor/results/safety_solver_judge_comparison/safety_input_index_to_harmmetric_prompt_id.csv")

# Raw files are only used in the optional text export cell below.
RAW_SOLVER_JSON_PATH = Path("benchmarks/safety/final_solver/all_attacks_all_models_scored.json")
HARMMETRIC_FINAL_DIR = Path("benchmarks/HarmMetric_Eval/Final files to use")

MISTRAL14_SOLVER_ID = "mistralai_ministral_3_14b_instruct_2512"
MISTRAL14_JUDGE_ID = "ministral_3_14b_instruct_2512_bf16"
QWEN2_SOLVER_ID = "qwen_qwen3_5_2b"
QWEN2_JUDGE_ID = "qwen3_5_2b"

# Tune these if you want stricter/looser examples.
JUDGE_SUCCESS_THRESHOLD = 0.75
JUDGE_FAILURE_THRESHOLD = 0.25
N_EXAMPLES_PER_GROUP = 8

print(OUT_DIR.resolve())

## Load Metadata-Only Matrices

This cell only loads matrices and metadata. It does not inspect prompt or response text.

In [ ]:
solver = pd.read_csv(SOLVER_MATRIX_PATH, index_col=0)
solver_item_meta = pd.read_csv(SOLVER_ITEM_META_PATH)
judge = pd.read_csv(JUDGE_MATRIX_PATH, index_col=0)
judge_item_meta = pd.read_csv(JUDGE_ITEM_META_PATH)
prompt_lookup = pd.read_csv(PROMPT_LOOKUP_PATH, usecols=["input_index", "source", "prompt_id"])

prompt_lookup["input_index"] = prompt_lookup["input_index"].astype(int)
prompt_lookup["prompt_id"] = prompt_lookup["prompt_id"].astype(str)
judge.columns = judge.columns.astype(str)

print("solver:", solver.shape)
print("judge:", judge.shape)
print("prompt lookup:", prompt_lookup.shape)
print("solver ids available:", [x for x in solver.index if "14b" in x or "2b" in x])
print("judge ids available:", [x for x in judge.index if "14b" in x or "2b" in x])

## Build Attack-Level Join

Each safety prompt has multiple attack-family variants. This table joins attack-level solver outcomes to prompt-level judge scores.

In [ ]:
def solver_attack_long(subject_id: str) -> pd.DataFrame:
    values = solver.loc[subject_id].rename("solver_score").reset_index()
    values = values.rename(columns={"index": "item_id"})
    out = values.merge(solver_item_meta, on="item_id", how="left")
    out["input_index"] = out["input_index"].astype(int)
    out = out.merge(prompt_lookup, on=["input_index", "source"], how="left", suffixes=("", "_lookup"))
    out["prompt_id"] = out["prompt_id"].astype(str)
    return out

def add_judge_scores(df: pd.DataFrame, judge_subject_id: str, score_col: str) -> pd.DataFrame:
    judge_scores = judge.loc[judge_subject_id].rename(score_col).reset_index()
    judge_scores = judge_scores.rename(columns={"index": "prompt_id"})
    judge_scores["prompt_id"] = judge_scores["prompt_id"].astype(str)
    return df.merge(judge_scores, on="prompt_id", how="left")

mistral14 = add_judge_scores(
    solver_attack_long(MISTRAL14_SOLVER_ID),
    MISTRAL14_JUDGE_ID,
    "judge_score",
)
mistral14["case_type"] = "mistral14_judge_success_solver_failure"

qwen2 = add_judge_scores(
    solver_attack_long(QWEN2_SOLVER_ID),
    QWEN2_JUDGE_ID,
    "judge_score",
)
qwen2["case_type"] = "qwen2_solver_success_judge_failure"

cols = ["case_type", "prompt_id", "item_id", "attack_family", "input_index", "source", "solver_score", "judge_score"]
print(mistral14[cols].head())
print(qwen2[cols].head())

## Select A Few Cases

Definitions:

- Mistral 14B judge-success / solver-failure: `judge_score >= 0.75` and `solver_score == 0.0`.
- Qwen3.5 2B solver-success / judge-failure: `solver_score == 1.0` and `judge_score <= 0.25`.

These are attack-level rows, so one prompt may appear under different attack families.

In [ ]:
mistral14_cases = (
    mistral14.loc[
        (mistral14["judge_score"] >= JUDGE_SUCCESS_THRESHOLD)
        & (mistral14["solver_score"] == 0.0),
        cols,
    ]
    .sort_values(["judge_score", "attack_family", "prompt_id"], ascending=[False, True, True])
    .head(N_EXAMPLES_PER_GROUP)
    .copy()
)

qwen2_cases = (
    qwen2.loc[
        (qwen2["solver_score"] == 1.0)
        & (qwen2["judge_score"] <= JUDGE_FAILURE_THRESHOLD),
        cols,
    ]
    .sort_values(["judge_score", "attack_family", "prompt_id"], ascending=[True, True, True])
    .head(N_EXAMPLES_PER_GROUP)
    .copy()
)

combined_cases = pd.concat([mistral14_cases, qwen2_cases], ignore_index=True)

mistral14_cases.to_csv(OUT_DIR / "safety_mistral14_judge_success_solver_failure_metadata_only.csv", index=False)
qwen2_cases.to_csv(OUT_DIR / "safety_qwen2_solver_success_judge_failure_metadata_only.csv", index=False)
combined_cases.to_csv(OUT_DIR / "safety_selected_solver_judge_case_studies_metadata_only.csv", index=False)

summary = pd.DataFrame([
    {"case_type": "mistral14_judge_success_solver_failure", "n_available": int(((mistral14["judge_score"] >= JUDGE_SUCCESS_THRESHOLD) & (mistral14["solver_score"] == 0.0)).sum()), "n_selected": len(mistral14_cases)},
    {"case_type": "qwen2_solver_success_judge_failure", "n_available": int(((qwen2["solver_score"] == 1.0) & (qwen2["judge_score"] <= JUDGE_FAILURE_THRESHOLD)).sum()), "n_selected": len(qwen2_cases)},
])
summary.to_csv(OUT_DIR / "safety_selected_solver_judge_case_studies_summary.csv", index=False)

print(summary)
display(combined_cases)

## Optional: Export Text Fields For Manual Inspection

This cell is off by default. Set `INCLUDE_TEXT = True` only when you want to save prompt/response text for your own inspection. It writes files but does not display the text in the notebook.

In [ ]:
INCLUDE_TEXT = False

if INCLUDE_TEXT:
    with RAW_SOLVER_JSON_PATH.open("r", encoding="utf-8") as f:
        raw_solver_rows = json.load(f)

    raw_solver_index = {}
    for row in raw_solver_rows:
        item_id = f"{row.get('attack_family')}:{int(row.get('input_index')):03d}"
        model = str(row.get("model", ""))
        raw_solver_index[(model, item_id)] = row

    harmmetric_files = {
        "mistral14": HARMMETRIC_FINAL_DIR / "Ministral-3-14B-Instruct-2512-BF16-HarmJudge-safety_solver_prompt_scores.csv",
        "qwen2": HARMMETRIC_FINAL_DIR / "Qwen3.5-2B-HarmJudge-safety_solver_prompt_scores.csv",
    }
    harmmetric_by_model = {}
    for key, path in harmmetric_files.items():
        # This intentionally reads the full file only in the optional cell.
        df = pd.read_csv(path)
        df["prompt_id"] = df["prompt_id"].astype(str)
        harmmetric_by_model[key] = df.set_index("prompt_id").to_dict("index")

    def attach_text(rows: pd.DataFrame, *, solver_model_contains: str, judge_key: str) -> pd.DataFrame:
        enriched = rows.copy()
        harmful_prompts = []
        solver_responses = []
        judge_harmful_prompts = []
        for _, row in enriched.iterrows():
            raw_solver = next(
                (
                    value
                    for (model, item_id), value in raw_solver_index.items()
                    if solver_model_contains in model and item_id == row["item_id"]
                ),
                {},
            )
            raw_judge = harmmetric_by_model[judge_key].get(str(row["prompt_id"]), {})
            harmful_prompts.append(raw_solver.get("harmful_prompt", ""))
            solver_responses.append(raw_solver.get("response", ""))
            judge_harmful_prompts.append(raw_judge.get("harmful_prompt", ""))
        enriched["solver_harmful_prompt"] = harmful_prompts
        enriched["solver_response"] = solver_responses
        enriched["judge_harmful_prompt"] = judge_harmful_prompts
        return enriched

    mistral14_text = attach_text(
        mistral14_cases,
        solver_model_contains="Ministral-3-14B",
        judge_key="mistral14",
    )
    qwen2_text = attach_text(
        qwen2_cases,
        solver_model_contains="Qwen3.5-2B",
        judge_key="qwen2",
    )
    combined_text = pd.concat([mistral14_text, qwen2_text], ignore_index=True)

    # Saved as JSON so long text fields stay readable when opened externally.
    combined_text.to_json(
        OUT_DIR / "safety_selected_solver_judge_case_studies_WITH_TEXT_FOR_MANUAL_REVIEW.json",
        orient="records",
        indent=2,
        force_ascii=False,
    )
    combined_text.drop(columns=["solver_harmful_prompt", "solver_response", "judge_harmful_prompt"]).to_csv(
        OUT_DIR / "safety_selected_solver_judge_case_studies_with_text_index.csv",
        index=False,
    )
    print("Wrote optional text export. Open the JSON manually if you want to inspect content.")
else:
    print("INCLUDE_TEXT is False; no prompt/response text was read or exported.")

## Note On `overall_effectiveness_score`

`overall_effectiveness_score` is not a binary accuracy variable, so `0.5` should not be described as random guessing by default. It is better to describe it as an intermediate rubric score. If we threshold it for case selection, the threshold is an analytic choice, not a chance-level interpretation.